# P0-2 — Per-layer eigenvalue spectrum of the (a − m) anchor calibration matrix

Top-to-bottom reproducer for `docs/insights/P0-2-eigenvalue-spectrum-evidence.md`.

**Outcome (graceful degradation branch).** Both pre-registered acceptance criteria FAIL:

- (a) Rank-8 elbow at L=26: `sv_7/sv_8 = 1.019` (threshold 1.5 — FAIL); `EV@K8 = 0.213` (threshold 0.70 — FAIL).
- (b) Effective rank monotonically decreases L=10 → final: **INVERTED** — Shannon eff_rank *increases* 1399 → 1713 (+22 %) over L=10 → L=26.

**Implication.** L=26 is the *maximum-dispersion* site, not the *integration-as-compression* site. K=8 is em-deal-breaker selected (§A.5 27-cell pilot grid), not spectrum-predicted. Paper §6.4 Insight 2 stays empirical; §5.2 / §6.6 routing-vs-integration framing softens to "consistent with high-rank residual encoding."

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pd.set_option('display.precision', 4)

## 1. (Re)build per-layer canonical CSV + figures

`scripts/analyze_e6_eigenvalue_spectrum.py` walks the OneVision E6 calibration dirs and emits
`docs/insights/_data/p0_2_per_layer_spectrum.{csv,md}` + `docs/figures/P0-2_*.png`. CPU-only, ~60 s.

In [ ]:
subprocess.run(
    ['uv', 'run', 'python', str(ROOT / 'scripts' / 'analyze_e6_eigenvalue_spectrum.py'),
     '--model', 'llava-onevision-qwen2-7b-ov',
     '--tags', 'plotqa,infographicvqa',
     '--K-probe', '8',
     '--peak-layer', '26'],
    cwd=ROOT, check=True)

## 2. Per-layer rank measures

Scale-invariant measures — needed because OneVision residual-stream norms grow ~50× with depth (otherwise "norm grew" confounds with "integration").

In [ ]:
spec = pd.read_csv(ROOT / 'docs/insights/_data/p0_2_per_layer_spectrum.csv')
highlight = spec[spec.layer.isin([0, 10, 14, 18, 22, 26, 27])]
cols = ['layer', 'sv_0', 'sv_7', 'sv_8',
        'sv7_over_sv8', 'explained_var_K8',
        'eff_rank_shannon', 'part_ratio', 'stable_rank']
print(highlight[cols].to_string(index=False))

## 3. Acceptance verdict (pre-registered thresholds)

Verdict written to `_data/p0_2_acceptance_verdict.json` by the analyzer. Both criteria fail:
- (a) `sv_7/sv_8 ≥ 1.5 OR EV@K8 ≥ 0.70`
- (b) Shannon `eff_rank` monotonically decreasing L=10 → L=27

In [ ]:
verdict = json.loads((ROOT / 'docs/insights/_data/p0_2_acceptance_verdict.json').read_text())
for k, v in verdict.items():
    print(f'  {k}: {v}')

## 4. Eigengap probe at K=8 across layers

If a rank-8 elbow existed at L=26, `sv_7/sv_8` should peak at L=26 and dominate `sv_8/sv_9` and other neighbouring ratios. Across all 28 layers `sv_7/sv_8 ∈ [1.02, 1.12]` and `sv_8/sv_9 ∈ [1.00, 1.13]` are comparable — no rank-8-localized gap at any layer.

In [ ]:
import torch
full_svs = torch.load(ROOT / 'docs/insights/_data/p0_2_full_svs_top.pt', weights_only=True)
rows = []
for L, S in sorted(full_svs.items()):
    rows.append({'layer': L,
                 'sv_6/sv_7': float(S[6] / S[7]),
                 'sv_7/sv_8': float(S[7] / S[8]),
                 'sv_8/sv_9': float(S[8] / S[9]),
                 'sv_9/sv_10': float(S[9] / S[10])})
ratios = pd.DataFrame(rows)
print(ratios.to_string(index=False))

## 5. Effective rank trajectory — INVERTED direction

Predicted: `eff_rank` decreases L=10 → L=27 (anchor variance concentrates at integration site).
Observed: `eff_rank` *increases* L=10 → L=26 (1399 → 1713) and only drops at L=27 where the LM head compresses for next-token logit projection.

L=26 = **maximum-dispersion** site, not minimum-dispersion site.

In [ ]:
rank_traj = spec[['layer', 'eff_rank_shannon', 'part_ratio', 'stable_rank', 'explained_var_K8']]
print(rank_traj.to_string(index=False))

## 6. Cross-link

- `docs/insights/P0-2-eigenvalue-spectrum-evidence.md` — full evidence + paper-update implications.
- `docs/insights/plan_post_review_2026-05-09.md` — Phase 5 priority queue (P0-2 entry).
- `docs/insights/_data/p0_2_per_layer_spectrum.csv` — canonical per-layer spectrum.
- `docs/insights/_data/p0_2_acceptance_verdict.json` — formal pass/fail.
- `docs/figures/P0-2_L26_spectrum.png` — Figure 1 (transparency item).
- `docs/figures/P0-2_per_layer_rank_trajectory.png` — Figure 2 (transparency item).
- `references/roadmap.md` §7 Phase 5 P0-2 row — status updated to FAIL-graceful-degradation.